# 05 Silver trades
Cleans bronze.trades, converts types, and sends rejected rows to a quarantine table.

In [0]:
from pyspark.sql import functions as F

clean = spark.table("bronze.trades")
for c in ["trade_id", "portfolio_id", "security_id", "trade_type", "currency"]:
    clean = clean.withColumn(c, F.expr(f"NULLIF(NULLIF(UPPER(TRIM({c})), ''), 'NULL')"))

In [0]:
REJECT_RULES = [
    ("MISSING_PORTFOLIO",   "portfolio_id IS NULL"),
    ("INVALID_TRADE_TYPE",  "trade_type IS NULL OR trade_type NOT IN ('BUY', 'SELL')"),
    ("INVALID_QUANTITY",    "TRY_CAST(quantity AS DECIMAL(18,4)) IS NULL OR TRY_CAST(quantity AS DECIMAL(18,4)) <= 0"),
    ("INVALID_PRICE",       "TRY_CAST(price AS DECIMAL(18,4)) IS NULL OR TRY_CAST(price AS DECIMAL(18,4)) <= 0"),
    ("INVALID_DATE",        "TRY_CAST(trade_date AS DATE) IS NULL OR TRY_CAST(settle_date AS DATE) IS NULL"),
    ("SETTLE_BEFORE_TRADE", "TRY_CAST(settle_date AS DATE) < TRY_CAST(trade_date AS DATE)"),
    ("INVALID_CURRENCY",    "currency IS NULL OR currency NOT IN ('USD', 'GBP', 'EUR')"),
    ("UNKNOWN_SECURITY",    "sec_found IS NULL"),
]
case_sql = "CASE " + " ".join(f"WHEN {cond} THEN '{reason}'" for reason, cond in REJECT_RULES) + " END"

In [0]:
known = (spark.table("bronze.securities")
         .select(F.upper(F.trim("security_id")).alias("sec_found"))
         .distinct())

checked = (clean.join(known, clean.security_id == known.sec_found, "left")
                .withColumn("reject_reason", F.expr(case_sql)))

In [0]:
good = checked.filter("reject_reason IS NULL")
bad  = checked.filter("reject_reason IS NOT NULL")

silver = good.select(
    "trade_id", "portfolio_id", "security_id",
    F.col("trade_date").cast("date"),
    F.col("settle_date").cast("date"),
    "trade_type",
    F.col("quantity").cast("decimal(18,4)"),
    F.col("price").cast("decimal(18,4)"),
    "currency",
    F.col("updated_at").cast("timestamp"),
    "_source_file", "_ingested_at",
)
silver.write.mode("overwrite").saveAsTable("silver.trades")
bad.drop("sec_found").write.mode("overwrite").saveAsTable("silver.trades_quarantine")

In [0]:
%sql
SELECT 'silver' AS tbl, COUNT(*) AS n FROM silver.trades
UNION ALL
SELECT 'quarantine', COUNT(*) FROM silver.trades_quarantine

In [0]:
%sql
SELECT trade_id, reject_reason FROM silver.trades_quarantine ORDER BY trade_id